# Automatic Validation of Hold-Out Test Sets with ROC-AUC Metric
This script builds on probe.ipynb, with the addition of a new automatic validation metric for example.txt and unrelated.txt.  
After multiple trials and evaluations of the automatic validation metric, I decided upon a final metric, which is the one used in this notebook.   
Please note that this final metric definitely is not perfect yet, and I would like to improve it further. However, out of all the options that I tried, I believe this metric provides the most complete and full-perspective automated validation (despite being a little harsh perhaps).   

For the purpose of the interview, I have added markdown sections with additional description/reasoning in this notebook.   

For the automatic validation, sentence transformers (SBERT) is used to provide an external 'benchmark' value for semantic textual similarity.   
Link to SBERT: https://sbert.net/index.html   
Link to explaination of semantic textual similarity: https://sbert.net/docs/sentence_transformer/usage/semantic_textual_similarity.html   

### Why was SBERT chosen?
SBERT was chosen over other semantic similiarity calculators (such as WuPalmer similarity) as it has the ability to compare similarity between not just words, but phrases and sentences. This is vital for our purpose of testing the probe, as it allows an input of token strings which consequently enables the validation of conceptual similarities, not just word similarity. 


In [1]:
# Install required libraries if not already installed (this currently isn't in requirements.txt)
# For installation: !pip install sentence_transformers

# Import libraries needed
import json
import numpy as np
import torch
from IPython.display import HTML, display
import html
import os
import nltk
from transformers import AutoTokenizer
from transformer_lens import HookedTransformer
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import roc_auc_score

# Download necessary NLTK data (for tokenization, if needed)
nltk.download('punkt')

# Load model once globally
def load_model(model_name="gemma-2-2b"):
    """Load the model and return it for reuse"""
    print(f"Loading model: {model_name}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = HookedTransformer.from_pretrained(model_name, device=device)
    return model, model.tokenizer, device

global_model, global_tokenizer, global_device = load_model()

# Define separators for grouping of tokens later
SEPARATORS = {'.', ',', '),', ').'}

def is_group_separator(token_strs, i):
    """
    Returns True if token_strs[i] is considered a separator for grouping.
    Tokens to be used as separators are: '.', ',', ').', '),'
    Note: '.' is not a separator if it is between digits - to avoid separating numbers.
    """
    token = token_strs[i].strip()
    if token not in SEPARATORS:
        return False
    # Check if '.' is between numeric tokens (e.g., "26.8")
    if token == '.':
        if i > 0 and i < len(token_strs) - 1:
            prev_tok = token_strs[i - 1].strip()
            next_tok = token_strs[i + 1].strip()
            if prev_tok.isdigit() and next_tok.isdigit():
                return False
    return True

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Loading model: gemma-2-2b


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model gemma-2-2b into HookedTransformer


## Separation of Tokens into Groups
### Why did I choose to separate into groups?
Originally I considered using each token separately, however it was clear that single tokens could not convey concepts by themselves as effectively as phrases/sentences could. Thus, a comparison between a single token and the concept seemed unfair and not effective for validation. I then considered using sentences, however sentences seemed too long and I thought that they weren't precise enough.    
Finally, I had the idea of separating by full-stops or commas, as the separation would create phrases which often express a single concept in language. Furthermore, I chose . or , as the activation most often occurs at them, due to the probe being trained on the final token.    
I wanted to make sure that each group only contained one . or , token, and so I chose to use them as separators.  

### How does the grouping work? 
The way the grouping works is that the tokens are grouped with the . or , that occurs after them (not before), due to the activation representing the concept in the tokens before (phase shift we talked about).    
For example: "He has high blood sugar, as he eats a lot of fast food." would be split into "He has high blood sugar," and "because he eats a lot of fast food."   

### Trouble shooting   
After I trialled this method, I saw that many of the phrases were not being split the way I imagined. After analysing the groups created and the tokens from the example text, I realised that this was because tokens such as ). and ), weren't being counted as separators. So, I added them to the separator list.

### Exceptions
I realised that numbers with decimals were being split into two groups due to the . token, so I added an exception for this case.

In [12]:
# For processing examples
def process_example(model, tokenizer, hook_name, probe, text, device):
    """Process a single example and return tokens and their activation scores."""
    # Tokenize the text
    tokens = tokenizer.encode(text, return_tensors="pt").to(device)

    # Get token strings
    token_strs = [tokenizer.decode(t).replace('▁', ' ') for t in tokens[0]]
    
    # Run model with cache to extract activations at the specified hook
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens, names_filter=[hook_name])
        activations = cache[hook_name]
    
    # Apply the probe to each token position
    scores = []
    for pos in range(activations.shape[1]):
        # Get activations for this position
        pos_activations = activations[0, pos].cpu().numpy().reshape(1, -1)

        # Apply the probe to get probability of positive class
        score = probe.predict_proba(pos_activations)[0, 1] 
        scores.append(float(score))
    
    return token_strs, scores

# Load model for semantic similarity
semantic_model = SentenceTransformer('all-MiniLM-L6-v2')

def compute_semantic_similarity(phrase, concept_text):
    """
    Compute cosine similarity between 'phrase' and 'concept_text' using Sentence-BERT.
    Returns a float in the range [-1, 1].
    """
    emb_phrase = semantic_model.encode(phrase, convert_to_tensor=True)
    emb_concept = semantic_model.encode(concept_text, convert_to_tensor=True)
    # Calculate cosine similarity and return as a float
    return util.cos_sim(emb_phrase, emb_concept).item()

# Calculate activation validation score by summing each token's contribution.
def compute_activation_validation_score(token_strs, scores, concept_text, is_example=True):
    """
    Calculate the activation validation score for a given text.
    
    For example text (is_example=True):
      - For each token group:
          if the group's semantic similarity > 0.10, add (activation * similarity)
          else, subtract (activation * (1 - similarity))
    For unrelated text (is_example=False):
      - For each token group, always subtract (activation * (1 - similarity))
      
    Returns the summed score across all token groups.
    """
    total = 0.0
    current_group_tokens = []
    current_group_scores = []
    
    for i, (token, score) in enumerate(zip(token_strs, scores)):
        # Accumulate tokens and their scores
        current_group_tokens.append(token)
        current_group_scores.append(score)
        
        # Process the group if the current token is a separator
        if is_group_separator(token_strs, i):
            group_string = "".join(current_group_tokens).strip()
            similarity = compute_semantic_similarity(group_string, concept_text)
            for sc in current_group_scores:
                if is_example:
                    if similarity > 0.10:
                        total += sc * similarity
                    else:
                        total -= sc * (1 - similarity)
                else:
                    total -= sc * (1 - similarity)
            # Reset the group lists
            current_group_tokens = []
            current_group_scores = []
    
    # Process any remaining tokens that didn't end with a separator
    if current_group_tokens:
        group_string = "".join(current_group_tokens).strip()
        similarity = compute_semantic_similarity(group_string, concept_text)
        for sc in current_group_scores:
            if is_example:
                if similarity > 0.10:
                    total += sc * similarity
                else:
                    total -= sc * (1 - similarity)
            else:
                total -= sc * (1 - similarity)
    
    return total

# Group tokens and return token-level details for printing.
def match_tokens_to_groups(token_strs, scores, concept_text, is_example=True):
    """
    Group tokens based on separators and calculate each token's contribution.
    
    For example text (is_example=True):
      - If group similarity > 0.10, contribution = activation * similarity.
      - Else, contribution = -activation * (1 - similarity).
    For unrelated text (is_example=False):
      - Contribution is always -activation * (1 - similarity).
      
    Returns a list of dictionaries with token details.
    """
    rows = []
    current_group_tokens = []
    current_group_indices = []
    current_group_scores = []
    
    for i, (token, score) in enumerate(zip(token_strs, scores)):
        # Add token, its score, and index to current group
        current_group_tokens.append(token)
        current_group_scores.append(score)
        current_group_indices.append(i)
        
        # If the token is a group separator, process the current group
        if is_group_separator(token_strs, i):
            group_string = "".join(current_group_tokens).strip()
            similarity = compute_semantic_similarity(group_string, concept_text)
            for idx, tok, sc in zip(current_group_indices, current_group_tokens, current_group_scores):
                if is_example:
                    if similarity > 0.10:
                        contribution = sc * similarity
                    else:
                        contribution = - sc * (1 - similarity)
                else:
                    contribution = - sc * (1 - similarity)
                rows.append({
                    "index": idx,
                    "token": tok,
                    "group": group_string,
                    "activation": sc,
                    "concept": concept_text,
                    "similarity": similarity,
                    "contribution": contribution
                })
            # Reset group lists
            current_group_tokens = []
            current_group_scores = []
            current_group_indices = []
    
    # Process any tokens left in the final group.
    if current_group_tokens:
        group_string = "".join(current_group_tokens).strip()
        similarity = compute_semantic_similarity(group_string, concept_text)
        for idx, tok, sc in zip(current_group_indices, current_group_tokens, current_group_scores):
            if is_example:
                if similarity > 0.10:
                    contribution = sc * similarity
                else:
                    contribution = - sc * (1 - similarity)
            else:
                contribution = - sc * (1 - similarity)
            rows.append({
                "index": idx,
                "token": tok,
                "group": group_string,
                "activation": sc,
                "concept": concept_text,
                "similarity": similarity,
                "contribution": contribution
            })
    return rows

# Print a text block table of token-level contributions.
def print_token_contributions_table(rows, label="Example Text"):
    """
    Print a fixed-width table showing token-level details.
    Columns: Index, Token, Group, Activation, Similarity, Contribution.
    Only tokens with activation above a threshold are shown.
    """
    threshold = 0.001  # Filter out tokens with very low activation
    filtered = [r for r in rows if abs(r["activation"]) > threshold]
    
    if not filtered:
        print(f"\nNo non-zero activations found for {label}.")
        return
    
    header_format = "{:<5} | {:<15} | {:<75} | {:>12} | {:>12} | {:>12}"
    line_width = 5 + 3 + 15 + 3 + 75 + 3 + 12 + 3 + 12 + 3 + 12
    header = header_format.format("Idx", "Token", "Group", "Activation", "Similarity", "Contribution")
    separator = "-" * line_width
    table_lines = [f"\n--- Token-Level Contributions ({label}) ---", header, separator]
    
    for r in filtered:
        idx_str = str(r["index"])
        token_str = r["token"].replace("\n", " ")  # Clean token string
        group_str = r["group"].replace("\n", " ")
        activation_str = f"{r['activation']:.4f}"
        similarity_str = f"{r['similarity']:.4f}"
        contribution_str = f"{r['contribution']:.4f}"
        line = header_format.format(
            idx_str,
            token_str[:15],
            group_str[:75],
            activation_str,
            similarity_str,
            contribution_str
        )
        table_lines.append(line)
    
    print("\n".join(table_lines))

## Activation Validation
### What is the semantic similarity calculated for?   
The semantic similarity is calculated between the concept and the group of tokens. As stated before, this is because the concept is better represented by a group of tokens.      

### What metrics are calculated for the activation validation?   
For each example text and unrelated text, a separate score is calculated that validates the probe's performance on that text. A total score is also calculated by summing all of the example and unrelated scores for a particular concept.  

### How is the activation validation score calculated for example texts?   
Example texts are designed to contain the concept tested.    
Each token contributes to the example score, with the contributions being calculated as follows:    
The initial score for an example score is set as 0.   
If the token's group semantic similarity is equal to or over 0.10, it is counted as a 'match'.
    For matches, add semantic similarity x activation to the score   
    This is a 'reward' that is used to reward the model for activating at a spot where the concept is present (according to semantic similarity)   
    The reward increases when semantic similarity is higher and activation is higher      
If the group semantic similarity is below 0.10, it is counted as a 'mismatch'.
    For mismatches, minus (1 - semantic similarity) x activation from the score
    This is a penalty    
    The penalty increases when semantic similarity is lower and when activation is higher   
Overall, this score represents a balance between false positives and true positives   
The bigger the score, the better.   

### How is the activation validation score calculated for unrelated texts?   
Unrelated texts do not contain the concepts, and so they should not result in any probe activation at all. 
Thus, the unrelated score is easier to calculate - follows a similar logic.   
Each token contributes to the unrelated score, with the contributions being calculated as follows:
The initial score for an unrelated score is set as 0.   
For all of the tokens, calculated (1 - semantic similarity) x activation and minus from the score. 
    Every token is counted as a mismatch
    This is a penalty    
    The penalty increases when semantic similarity is lower and when activation is higher   
The score will always be 0 or negative.   
The closer the score is to 0, the better it is (0 represents no activation at all)   
- A lower (more negative) score represents more false positives   

### How is the total validation score calculated?   
The total score is just the sum of the example and unrelated scores.   
The higher score the better.     

### Why not use absolute semantic similarity? 
Absolute semantic similarity is used as semantic similarity ranges from -1 to 1. 
A score of 0 represents that the phrases are completely unrelated. 1 means they are perfectly aligned in meaning, where are -1 represents that they are opposite in meaning.   
After analysing the examples further, I realised that the negative semantic similarities do have important meanings and that we are wanting to design a probe that can detect precise enough definitions. Thus, I have chosen to ultimately keep the semantic similarity as it is.   

### Why use multiplication for the metric (rather than differences or sums)? 
I debated between using the multiplication of variables vs using other methods like sums or differences. In the end, I chose multiplication due to several reasons:   
- Tokens with activation of 0 would result in an 0 for the validation score - this helps focus on the tokens that are activated and reduces calculation complexity   
- Multiplication allows the metric to have a proportional relationship with its constituent variables - this allows the activation/similarity to proportionately affect the metric
- The multiplication focuses more on the synergy between the similarity and the activation, which makes sense as we are using the similarity as an external benchmark...   
- Finally, the scaling for semantic similarity and activation is very different and so differences/sums would not provide a reliable metric   

### Weaknesses and Future Directions    
Right now, certain groups of tokens are often summed more than once due to there being multiple tokens afterwards being activated (e.g. /n, space, .). They do have different activations (which is why I chose to keep it), but a future direction would be to see if it would be better to just count each group once (as it's the same concept).

In addition, as you might have noticed, this metric is only focusing on the false positives and true positives. So, it does not provide a good evaluation of the true and false negatives.    
This is because, for negatives, the activation is 0 --> so it does not contribute to the score.   
It would be worthwhile to explore adding another part to the metric calculation - something symmetrical.   
For example, to penalise false negatives:     
If the activation is below a threshold, minus (1 - activation) x semantic similarity to the score   
- This should penalise false negatives, but not true negatives   

Finally, I've found that sometimes this score is a bit too harsh
- Doesn't fully match up with the probe performance sometimes (human eye)    
- Due to all the penalities - can add up a lot

In [13]:
# Compute ROC AUC using token-level activation scores (not activation validation)
def compute_token_level_roc_auc_new(example_rows, unrelated_rows, label_threshold=0.5):
    """
    Compute ROC AUC for token-level activation scores (raw probe outputs).
    
    Labeling Method:
    - Find the first token in example text where activation exceeds label_threshold.
    - Tokens before this token are labeled as 0 (negative).
    - Tokens at or after this token are labeled as 1 (positive).
    - All tokens from the unrelated text are labeled as 0.
    
    Uses raw activations to measure the probe’s effectiveness at detecting the concept.
    """
    # Sort example rows by token index
    example_sorted = sorted(example_rows, key=lambda r: r["index"])
    
    # Find the first token (mark token) where activation exceeds the threshold
    mark_index = None
    for r in example_sorted:
        if r["activation"] > label_threshold:
            mark_index = r["index"]
            break
    if mark_index is None:
        print("No token in example text reaches the threshold; cannot compute ROC AUC.")
        return None
    
    # Label example tokens: tokens with index >= mark_index get label 1; those before get label 0.
    y_example = [1 if r["index"] >= mark_index else 0 for r in example_sorted]
    scores_example = [r["activation"] for r in example_sorted] 
    
    # For unrelated text, label all tokens as 0.
    unrelated_sorted = sorted(unrelated_rows, key=lambda r: r["index"])
    y_unrelated = [0] * len(unrelated_sorted)
    scores_unrelated = [r["activation"] for r in unrelated_sorted] 
    
    # Combine labels and scores from both texts
    y_true = y_example + y_unrelated
    y_scores = scores_example + scores_unrelated
    
    return roc_auc_score(y_true, y_scores)

# Validate example and unrelated texts, then print overall and token-level details.
def validate_multiple_texts(concept_key, example_files, unrelated_files, concept_string=None, layer=22):
    """
    1) For each example file, process the text, compute the validation score, and print its table.
    2) For each unrelated file, process the text, compute the validation score, and print its table.
    3) Compute the combined overall score (sum of scores from all example files plus sum from all unrelated files).
    4) Combine all token-level rows from examples and unrelated texts and compute ROC AUC.
    5) Return a dictionary containing individual scores and the overall score.
    """
    if concept_string is None:
        concept_string = concept_key.replace("_", " ")
    
    overall_example_score = 0.0
    overall_unrelated_score = 0.0
    example_file_scores = {}
    unrelated_file_scores = {}
    all_example_rows = []
    all_unrelated_rows = []
    
    # Set up probe file paths (assumed same for all files for a concept)
    probe_dir = os.path.join("experiment_probes/p25", concept_key)
    joblib_path = os.path.join(probe_dir, "probe.joblib")
    pkl_path = os.path.join(probe_dir, "probe.pkl")
    config_path = os.path.join(probe_dir, "config.json")
    
    if os.path.exists(joblib_path):
        import joblib
        probe = joblib.load(joblib_path)
        print(f"Loaded probe from {joblib_path}")
    elif os.path.exists(pkl_path):
        import pickle
        with open(pkl_path, 'rb') as f:
            probe = pickle.load(f)
        print(f"Loaded probe from {pkl_path}")
    else:
        print(f"Probe not found at {joblib_path} or {pkl_path}")
        return None
    
    if os.path.exists(config_path):
        with open(config_path, "r") as f:
            config = json.load(f)
        concept_string = config.get("concept", concept_string)
    
    hook_name = f"blocks.{layer}.hook_resid_post"
    
    # Process each example file
    for file in example_files:
        with open(file, "r", encoding="utf-8") as f:
            text = f.read().strip()
        tokens, scores = process_example(global_model, global_tokenizer, hook_name, probe, text, global_device)
        score = compute_activation_validation_score(tokens, scores, concept_string, is_example=True)
        example_file_scores[file] = score
        overall_example_score += score
        print(f"\n--- Validation for {file} (Example) ---")
        rows = match_tokens_to_groups(tokens, scores, concept_string, is_example=True)
        print_token_contributions_table(rows, label=f"Example: {file}")
    
    # Process each unrelated file
    for file in unrelated_files:
        with open(file, "r", encoding="utf-8") as f:
            text = f.read().strip()
        tokens, scores = process_example(global_model, global_tokenizer, hook_name, probe, text, global_device)
        score = compute_activation_validation_score(tokens, scores, concept_string, is_example=False)
        unrelated_file_scores[file] = score
        overall_unrelated_score += score
        print(f"\n--- Validation for {file} (Unrelated) ---")
        rows = match_tokens_to_groups(tokens, scores, concept_string, is_example=False)
        print_token_contributions_table(rows, label=f"Unrelated: {file}")
    
    # Combine overall score (total of example scores + total of unrelated scores)
    overall_score = overall_example_score + overall_unrelated_score
    
    # Combine all token-level rows for ROC AUC computation
    for file in example_files:
        with open(file, "r", encoding="utf-8") as f:
            text = f.read().strip()
        tokens, scores = process_example(global_model, global_tokenizer, hook_name, probe, text, global_device)
        rows = match_tokens_to_groups(tokens, scores, concept_string, is_example=True)
        all_example_rows.extend(rows)
    for file in unrelated_files:
        with open(file, "r", encoding="utf-8") as f:
            text = f.read().strip()
        tokens, scores = process_example(global_model, global_tokenizer, hook_name, probe, text, global_device)
        rows = match_tokens_to_groups(tokens, scores, concept_string, is_example=False)
        all_unrelated_rows.extend(rows)
    
    roc_auc = compute_token_level_roc_auc_new(all_example_rows, all_unrelated_rows, label_threshold=0.5)
    
    # Print overall results for the concept
    print("\n=== Combined Validation Results ===")
    print(f"Concept Key: {concept_key}")
    print(f"Concept String: '{concept_string}'")
    print("\n--- Individual Example Scores ---")
    for file, score in example_file_scores.items():
        print(f"{file}: {score:.4f}")
    print("\n--- Individual Unrelated Scores ---")
    for file, score in unrelated_file_scores.items():
        print(f"{file}: {score:.4f}")
    print(f"\nOverall Example Score:   {overall_example_score:.4f}")
    print(f"Overall Unrelated Score: {overall_unrelated_score:.4f}")
    print(f"Combined Overall Score:  {overall_score:.4f}")
    if roc_auc is not None:
        print(f"Token-level ROC AUC: {roc_auc:.4f}")
    
    # Prepare results dictionary for final output file (only scores, not full token details)
    results = {
        "example_file_scores": example_file_scores,
        "unrelated_file_scores": unrelated_file_scores,
        "overall_example_score": overall_example_score,
        "overall_unrelated_score": overall_unrelated_score,
        "combined_overall_score": overall_score,
        "token_roc_auc": roc_auc
    }
    
    return results

## ROC-AUC Metric Explanation
In our script, each token is treated as a sample. For the labelling, we label all the tokens in the unrelated texts as 0. For the tokens in the example texts, we use a different method. We first find the token where activation first exceeds a configurable threshold (default 0.5). Tokens before that token get label 0 and tokens at/after that token get label 1. Note, the activation threshold can be changed. 

The token’s activation serves as the classifier’s score. The idea is that tokens with high scores should belong to the example text (positive) and those with low scores should belong to the unrelated text (negative). 

By comparing the distribution of these scores for tokens labeled as 1 versus those labeled as 0 across various thresholds, the ROC curve is generated. The AUC quantifies how well these two groups are separated. A higher AUC means that the activation validation scores are effective at distinguishing tokens from the two texts.   

Note: I thought about using the activation validation as the classifier's score, however I think that using activation validation (which includes similarity) could make the evaluation dependent on semantic similarity, rather than just on the probe's effectiveness.

### Note on Results
Overall results are saved to probes_results

In [14]:
# List available concepts from JSON file
def list_available_concepts(json_file_path):
    """
    List all available concepts in the JSON file.
    
    Parameters:
    -----------
    json_file_path : str
        Path to the JSON file containing the concepts
    
    Returns:
    --------
    List of available concepts
    """
    with open(json_file_path, 'r') as file:
        data = json.load(file)
    concepts = data['concepts']
    return [concept.replace(" ", "_") for concept in concepts]

# Visualisation Function
def visualize_concept_on_text(text, concept_key, model=global_model, tokenizer=global_tokenizer, layer=22):
    """
    Create an HTML visualization of token-level activations for a concept on user input text.
    
    Args:
        text: The text to analyze
        concept_key: The concept key (e.g., 'elevated_LDL_cholesterol')
        model: Pre-loaded model (uses global model by default)
        tokenizer: Pre-loaded tokenizer
        layer: Layer to extract representations from
    """
    # Load the probe
    probe_dir = os.path.join("experiment_probes/p25", concept_key)
    joblib_path = os.path.join(probe_dir, "probe.joblib")
    pkl_path = os.path.join(probe_dir, "probe.pkl")
    config_path = os.path.join(probe_dir, "config.json")
    
    # Check for both joblib and pkl files
    if os.path.exists(joblib_path):
        import joblib
        probe = joblib.load(joblib_path)
        print(f"Loaded probe from {joblib_path}")
    elif os.path.exists(pkl_path):
        import pickle
        with open(pkl_path, 'rb') as f:
            probe = pickle.load(f)
        print(f"Loaded probe from {pkl_path}")
    else:
        print(f"Probe not found at {joblib_path} or {pkl_path}")
        return None
    
    # Load the config to get the concept name
    if not os.path.exists(config_path):
        print(f"Config not found at {config_path}")
        return None
    
    with open(config_path, "r") as f:
        config = json.load(f)
    
    concept = config.get("concept", concept_key.replace("_", " "))
    
    # Hook name for the residual stream at the specified layer
    hook_name = f"blocks.{layer}.hook_resid_post"
    
    # Process user input text
    tokens, scores = process_example(model, tokenizer, hook_name, probe, text, global_device)
    
    # Create HTML output
    html_output = f"<h2>Activation visualization for concept: '{concept}'</h2>"
    html_output += "<div style='line-height: 2.5; font-family: monospace; font-size: 14px;'>"
    
    for i, (token, score) in enumerate(zip(tokens, scores)):
        # Escape HTML special characters
        escaped_token = html.escape(token)

        # Calculate color intensity based on activation
        green_intensity = 255
        other_intensity = int(255 * (1 - score))
        color = f"rgb({other_intensity}, {green_intensity}, {other_intensity})"

        # Create token span
        html_output += f"""<span title='Token: "{escaped_token}"
Position: #{i}
Activation: {score:.4f}' style='background-color: {color}; padding: 3px; border-radius: 3px; margin: 1px;'>{escaped_token}</span>"""
    
    html_output += "</div>"
    # Add color scale
    html_output += """
    <div style='margin-top: 20px;'>
        <h3>Color Scale</h3>
        <div style='display: flex; width: 500px;'>
            <span style='background-color: rgb(255, 255, 255); width: 100px; padding: 10px; text-align: center;'>0.0</span>
            <span style='background-color: rgb(192, 255, 192); width: 100px; padding: 10px; text-align: center;'>0.25</span>
            <span style='background-color: rgb(128, 255, 128); width: 100px; padding: 10px; text-align: center;'>0.5</span>
            <span style='background-color: rgb(64, 255, 64); width: 100px; padding: 10px; text-align: center;'>0.75</span>
            <span style='background-color: rgb(0, 255, 0); width: 100px; padding: 10px; text-align: center;'>1.0</span>
        </div>
    </div>
    """
    return HTML(html_output)

# Main usage example
# Load example and unrelated texts from multiple files
example_files = [f"inputs/example{i}.txt" for i in range(1, 7)]  ### NEW: 6 example files
unrelated_files = [f"inputs/unrelated{i}.txt" for i in range(1, 7)]  ### NEW: 6 unrelated files

# Load concepts
concepts = list_available_concepts("inputs/concepts_exp.json")

# Dictionary to store final results for each concept (only scores, not token details)
all_results = {}

for concept in concepts:
    print("\n============================================")
    print(f"Concept: {concept}")
    
    print("\n--- Visualizing Example Texts ---")
    for file in example_files:
        with open(file, "r", encoding="utf-8") as f:
            text = f.read()
        display(visualize_concept_on_text(text, concept))
    
    print("\n--- Visualizing Unrelated Texts ---")
    for file in unrelated_files:
        with open(file, "r", encoding="utf-8") as f:
            text = f.read()
        display(visualize_concept_on_text(text, concept))
    
    print("\n--- Validation Results ---")
    # Validate all texts for this concept using the new function
    results = validate_multiple_texts(concept, example_files, unrelated_files)
    if results is not None:
        all_results[concept] = results

# Reorder concepts by combined overall score (largest to smallest)
sorted_concepts = sorted(all_results.items(), key=lambda x: x[1]["combined_overall_score"], reverse=True)

print("\n=== Concepts Ranked by Combined Overall Score (Highest to Lowest) ===")
for concept, res in sorted_concepts:
    print(f"{concept}: {res['combined_overall_score']:.4f}")

# Save final results to JSON file (only scores, not individual token activations)
output_json_path = "probes_results/p25.json"
with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2)
print(f"\nAll results saved to: {output_json_path}")


Concept: high_total_cholesterol

--- Visualizing Example Texts ---
Loaded probe from experiment_probes/p25/high_total_cholesterol/probe.joblib


Loaded probe from experiment_probes/p25/high_total_cholesterol/probe.joblib


Loaded probe from experiment_probes/p25/high_total_cholesterol/probe.joblib


Loaded probe from experiment_probes/p25/high_total_cholesterol/probe.joblib


Loaded probe from experiment_probes/p25/high_total_cholesterol/probe.joblib


...

Loaded probe from experiment_probes/p25/hypothyroidism/probe.joblib


Loaded probe from experiment_probes/p25/hypothyroidism/probe.joblib


Loaded probe from experiment_probes/p25/hypothyroidism/probe.joblib


Loaded probe from experiment_probes/p25/hypothyroidism/probe.joblib



--- Validation Results ---
Loaded probe from experiment_probes/p25/hypothyroidism/probe.joblib

--- Validation for inputs/example1.txt (Example) ---

--- Token-Level Contributions (Example: inputs/example1.txt) ---
Idx   | Token           | Group                                                                       |   Activation |   Similarity | Contribution
--------------------------------------------------------------------------------------------------------------------------------------------------
25    |  fatigue        | presents for a routine check-up with occasional headaches and fatigue persi |       0.9996 |       0.2008 |       0.2007
31    | .               | presents for a routine check-up with occasional headaches and fatigue persi |       0.0529 |       0.2008 |       0.0106
73    |  hydration      | worsening by end of the workday but relieved with rest and hydration.       |       0.0012 |       0.1450 |       0.0002
74    | .               | worsening by end of the

## How to use activation validation score?
Firstly, whilst this is an automated validation method, some level of human observation is still required. Overall, I would say that a difference of 0.5 above between the example and unrelated text score is at least required to show that there is a difference in the probe's activity. However, this is really varied depending on what the example text score actually is and also how related the example text itself is to the concept.    
It is benefitical to look at the ratio of the example and unrelated text scores and also to look at the individual activations as well.   
Note: I also tried validating my score using ChatGPT to rank the concepts within the example text, however this did not work very well...